In [1]:
from tqdm.auto import tqdm
import pandas as pd
import polars as pl
import numpy as np
import argparse
import os
import glob
import sys
import math

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from typing import Optional, Tuple, List, Dict, Any



In [2]:
DATA_DIR = "/kaggle/input/hms-harmful-brain-activity-classification/"
SPEC_DIR = os.path.join(DATA_DIR, "test_spectrograms/")
EEG_DIR = os.path.join(DATA_DIR, "test_eegs/")

df_train = pl.read_csv(os.path.join(DATA_DIR, "train.csv"))
df_test = pl.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_submission = pl.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

df_test.head()



spectrogram_id,eeg_id,patient_id
i64,i64,i64
2207717,2578018731,34153
2207717,2578018731,34153
2207717,2578018731,34153
2207717,2578018731,34153
8352559,1706196508,37552


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)




Using device: cuda


In [4]:
def load_model(path: str, model: nn.Module) -> nn.Module:
    ckpt = torch.load(path, map_location=torch.device("cpu"))
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        sd = ckpt["model_state_dict"]
    else:
        sd = ckpt
    model.load_state_dict(sd, strict=True)
    return model




In [5]:
HAVE_EXTERNAL_MODELS = False
MultimodalModel = None

try:
    sys.path.append("/kaggle/input/hms-models/")
    from eeg_cnn_rnn_att import EegModel as EegModel  # noqa: F401
    from spc_cnn_att import SpectrogramCnnModel as SpcModel  # noqa: F401
    from comb_model import MultimodalModel as _MultimodalModel

    MultimodalModel = _MultimodalModel
    HAVE_EXTERNAL_MODELS = True
except Exception as e:
    print("Warning: external models not available; using baseline predictions.")
    print("Import error:", repr(e))



Import error: ModuleNotFoundError("No module named 'eeg_cnn_rnn_att'")


In [6]:
models_multimodal = []
if HAVE_EXTERNAL_MODELS:
    for model_path in sorted(
        glob.glob(
            "/kaggle/input/hms-models/final_multimodal_sanity/final_multimodal_sanity/*"
        )
    ):
        try:
            model = MultimodalModel(None, None, None)
            print("Loading:", model_path)
            model = load_model(model_path, model)
            model = model.to(device)
            model.eval()
            models_multimodal.append(model)
        except Exception as e:
            print("Corrupted or incompatible:", model_path, "err:", repr(e))

print("Loaded multimodal models:", len(models_multimodal))




Loaded multimodal models: 0


In [7]:
def MAD(signal, axis=-1):
    """Compute the robust standard deviation (MAD) of a signal."""
    median = np.median(signal, axis=axis, keepdims=True)
    absolute_deviations = np.abs(signal - median)
    median_absolute_deviation = np.median(absolute_deviations, axis=axis, keepdims=True)
    scale_factor = 1.4826  # constant for normal distribution
    robust_std = median_absolute_deviation * scale_factor
    return robust_std


from scipy.signal import butter, filtfilt


def butter_filter(eeg_data, fs=200, cutoff_freq=22, order=4, btype="lowpass"):
    b, a = butter(N=order, Wn=cutoff_freq / (0.5 * fs), btype=btype, analog=False)
    return filtfilt(b, a, eeg_data)




In [8]:
import scipy  # noqa: F401
import torchaudio
from torchaudio.transforms import Spectrogram as _Spectrogram

n_fft = 800
win_length = 256
hop_length = 44

spectrogram = _Spectrogram(
    n_fft=n_fft, win_length=win_length, hop_length=hop_length, power=None
)


@torch.no_grad()
def compute_spec(chain):
    chain = torch.as_tensor(chain, dtype=torch.float32)
    chain = spectrogram(chain)  # (C, F, TT), complex
    chain = chain[:, :, 2:98]  # crop time bins
    chain = torch.abs(chain) / 15.0
    chain = torch.log(chain.clamp(min=math.exp(-4), max=math.exp(7)))
    chain = chain.mean(axis=1)  # (C, TT)
    return chain.cpu().numpy()


def compute_spec_eeg(a, b) -> np.ndarray:
    return butter_filter(
        a - b, cutoff_freq=np.array([0.25, 40.0]), order=5, btype="bandpass"
    )


def compute_spec_chain(df_eeg: pl.DataFrame) -> np.ndarray:
    Fp1 = df_eeg["Fp1"].to_numpy()
    Fp2 = df_eeg["Fp2"].to_numpy()
    F7 = df_eeg["F7"].to_numpy()
    F8 = df_eeg["F8"].to_numpy()
    T3 = df_eeg["T3"].to_numpy()
    T4 = df_eeg["T4"].to_numpy()
    T5 = df_eeg["T5"].to_numpy()
    T6 = df_eeg["T6"].to_numpy()
    F3 = df_eeg["F3"].to_numpy()
    F4 = df_eeg["F4"].to_numpy()
    C3 = df_eeg["C3"].to_numpy()
    C4 = df_eeg["C4"].to_numpy()
    P3 = df_eeg["P3"].to_numpy()
    P4 = df_eeg["P4"].to_numpy()
    O1 = df_eeg["O1"].to_numpy()
    O2 = df_eeg["O2"].to_numpy()

    ll = np.stack(
        [
            (
                compute_spec_eeg(Fp1, F7),
                compute_spec_eeg(F7, T3),
                compute_spec_eeg(T3, T5),
                compute_spec_eeg(T5, O1),
            )
        ]
    )
    lp = np.stack(
        [
            (
                compute_spec_eeg(Fp1, F3),
                compute_spec_eeg(F3, C3),
                compute_spec_eeg(C3, P3),
                compute_spec_eeg(P3, O1),
            )
        ]
    )
    rp = np.stack(
        [
            (
                compute_spec_eeg(Fp2, F4),
                compute_spec_eeg(F4, C4),
                compute_spec_eeg(C4, P4),
                compute_spec_eeg(P4, O2),
            )
        ]
    )
    rl = np.stack(
        [
            (
                compute_spec_eeg(Fp2, F8),
                compute_spec_eeg(F8, T4),
                compute_spec_eeg(T4, T6),
                compute_spec_eeg(T6, O2),
            )
        ]
    )

    chain = np.stack([ll, lp, rp, rl])[:, 0]  # (4, T)

    mads = MAD(chain, axis=-1)
    mads = np.median(mads.reshape(-1))
    chain = chain / (mads + 1e-5)

    chain = compute_spec(chain)  # (4, TT)
    return chain


def compute_spec_from_file(filepath: str) -> np.ndarray:
    df_eeg = pl.read_parquet(filepath).fill_null(0)
    chain = compute_spec_chain(df_eeg)
    return chain




In [9]:
def process_spec(spec: np.ndarray) -> np.ndarray:
    spec = spec[:, 1:]
    spec = np.stack(
        [
            spec[:, 0:100].T,
            spec[:, 100:200].T,
            spec[:, 200:300].T,
            spec[:, 300:400].T,
        ]
    )
    return spec


def compute_kaggle_spec_from_file(filepath: str) -> np.ndarray:
    spec = pl.read_parquet(filepath).to_numpy().astype(np.float32)
    spec = process_spec(spec)
    return spec




In [10]:
from typing import Union


def bin_array(
    array,
    bin_size,
    axis=-1,
    pad_dir="symmetric",
    mode="edge",
    return_padding=False,
    **padding_kwargs,
) -> Union[np.ndarray, Tuple[np.ndarray, List]]:
    if axis == -1:
        axis = array.ndim - 1

    curr_len = array.shape[axis]
    n_bins = math.ceil(curr_len / bin_size)
    new_len = n_bins * bin_size

    new_shape = list(array.shape)
    new_shape[axis] = n_bins
    new_shape.insert(axis + 1, bin_size)

    padding = [(0, 0)] * array.ndim
    if curr_len != new_len:
        if pad_dir == "left":
            pad_l = new_len - curr_len
            pad_r = 0
        elif pad_dir == "right":
            pad_l = 0
            pad_r = new_len - curr_len
        else:
            pad_l = (new_len - curr_len) // 2
            pad_r = (new_len - curr_len) - pad_l

        padding[axis] = (pad_l, pad_r)
        array = np.pad(array, padding, mode=mode, **padding_kwargs)

    array = array.reshape(new_shape)

    if return_padding:
        return array, padding

    return array




In [11]:
def compute_eeg(eeg: np.ndarray) -> np.ndarray:
    eeg = butter_filter(eeg, cutoff_freq=np.array([0.25, 50]), btype="bandpass")
    eeg = bin_array(eeg, bin_size=4, mode="reflect").mean(axis=-1)
    return eeg


def compute_eeg_chain(df_eeg: pl.DataFrame) -> np.ndarray:
    Fp1 = df_eeg["Fp1"].to_numpy()
    Fp2 = df_eeg["Fp2"].to_numpy()
    Fz = df_eeg["Fz"].to_numpy()
    Cz = df_eeg["Cz"].to_numpy()
    Pz = df_eeg["Pz"].to_numpy()
    F3 = df_eeg["F3"].to_numpy()
    F4 = df_eeg["F4"].to_numpy()
    F7 = df_eeg["F7"].to_numpy()
    F8 = df_eeg["F8"].to_numpy()
    C3 = df_eeg["C3"].to_numpy()
    C4 = df_eeg["C4"].to_numpy()
    P3 = df_eeg["P3"].to_numpy()
    P4 = df_eeg["P4"].to_numpy()
    T3 = df_eeg["T3"].to_numpy()
    T4 = df_eeg["T4"].to_numpy()
    T5 = df_eeg["T5"].to_numpy()
    T6 = df_eeg["T6"].to_numpy()
    O1 = df_eeg["O1"].to_numpy()
    O2 = df_eeg["O2"].to_numpy()

    # Keep the EKG fix: use the actual EKG channel when present, otherwise fall back safely.
    if "EKG" in df_eeg.columns:
        ekg = df_eeg["EKG"].to_numpy()
    else:
        ekg = df_eeg["O2"].to_numpy()

    ekg = butter_filter(ekg, cutoff_freq=np.array([0.50, 20.0]), btype="bandpass")
    ekg = bin_array(ekg, bin_size=4, mode="reflect").mean(axis=-1)
    ekg = ekg.reshape(1, -1)

    ll = np.stack(
        [
            (
                compute_eeg(Fp1 - F7),
                compute_eeg(F7 - T3),
                compute_eeg(T3 - T5),
                compute_eeg(T5 - O1),
            )
        ]
    )
    lp = np.stack(
        [
            (
                compute_eeg(Fp1 - F3),
                compute_eeg(F3 - C3),
                compute_eeg(C3 - P3),
                compute_eeg(P3 - O1),
            )
        ]
    )
    rp = np.stack(
        [
            (
                compute_eeg(Fp2 - F4),
                compute_eeg(F4 - C4),
                compute_eeg(C4 - P4),
                compute_eeg(P4 - O2),
            )
        ]
    )
    rl = np.stack(
        [
            (
                compute_eeg(Fp2 - F8),
                compute_eeg(F8 - T4),
                compute_eeg(T4 - T6),
                compute_eeg(T6 - O2),
            )
        ]
    )
    mid = np.stack([compute_eeg(Fz - Cz), compute_eeg(Cz - Pz)])

    chain = np.stack([ll, lp, rp, rl])[:, 0]
    return chain, mid, ekg


def compute_eeg_from_file(filepath: str) -> np.ndarray:
    df_eeg = pl.read_parquet(filepath).fill_null(0)
    return compute_eeg_chain(df_eeg)




In [12]:
LABELS = [
    "seizure_vote",
    "lpd_vote",
    "gpd_vote",
    "lrda_vote",
    "grda_vote",
    "other_vote",
]




In [13]:
def _safe_row_normalize(p: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    p = np.asarray(p, dtype=np.float64)
    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    p = np.clip(p, eps, None)
    s = np.clip(p.sum(axis=-1, keepdims=True), eps, None)
    return p / s


def _compute_global_prior_from_train(
    df_train: pl.DataFrame, alpha: float = 1.0
) -> np.ndarray:
    votes = df_train.select(LABELS).to_numpy().astype(np.float64)
    votes = np.nan_to_num(votes, nan=0.0, posinf=0.0, neginf=0.0)
    votes = np.clip(votes, 0.0, None)
    row_sums = np.clip(votes.sum(axis=1, keepdims=True), 1.0, None)
    probs = votes / row_sums
    prior = probs.mean(axis=0)
    prior = prior + alpha
    prior = _safe_row_normalize(prior).reshape(-1)
    return prior


def _compute_record_level_group_priors_and_counts(
    df_train: pl.DataFrame,
    group_col: str,
    global_prior: np.ndarray,
    alpha: float = 18.0,
) -> Tuple[Dict[int, np.ndarray], Dict[int, float]]:
    base = (
        df_train.group_by([group_col, "label_id"])
        .agg([pl.col(c).sum().alias(c) for c in LABELS])
        .with_columns(pl.sum_horizontal([pl.col(c) for c in LABELS]).alias("_vote_sum"))
        .group_by(group_col)
        .agg(
            [pl.col(c).sum().alias(c) for c in LABELS]
            + [pl.col("_vote_sum").sum().alias("_vote_sum")]
        )
    )

    ids = base[group_col].to_list()
    vote_mat = base.select(LABELS).to_numpy().astype(np.float64)
    vote_mat = np.nan_to_num(vote_mat, nan=0.0, posinf=0.0, neginf=0.0)
    vote_mat = np.clip(vote_mat, 0.0, None)

    evidence_mass = base["_vote_sum"].to_list()
    evidence = {int(k): float(v) for k, v in zip(ids, evidence_mass)}

    priors: Dict[int, np.ndarray] = {}
    counts: Dict[int, float] = {}

    for gid, v in zip(ids, vote_mat):
        gid_int = int(gid)
        p = v + alpha * global_prior
        p = _safe_row_normalize(p).reshape(-1)
        priors[gid_int] = p
        counts[gid_int] = float(evidence.get(gid_int, 0.0))

    return priors, counts


GLOBAL_PRIOR = _compute_global_prior_from_train(df_train, alpha=1.0)

EEG_PRIORS, EEG_COUNT = _compute_record_level_group_priors_and_counts(
    df_train, "eeg_id", GLOBAL_PRIOR, alpha=18.0
)
SPEC_PRIORS, SPEC_COUNT = _compute_record_level_group_priors_and_counts(
    df_train, "spectrogram_id", GLOBAL_PRIOR, alpha=18.0
)
PATIENT_PRIORS, PATIENT_COUNT = _compute_record_level_group_priors_and_counts(
    df_train, "patient_id", GLOBAL_PRIOR, alpha=18.0
)

print("GLOBAL_PRIOR:", GLOBAL_PRIOR, "sum:", float(GLOBAL_PRIOR.sum()))
print("Num eeg priors:", len(EEG_PRIORS))
print("Num spectrogram priors:", len(SPEC_PRIORS))
print("Num patient priors:", len(PATIENT_PRIORS))



GLOBAL_PRIOR: [0.17268109 0.16164074 0.16061184 0.16318858 0.16864811 0.17322963] sum: 1.0000000000000002
Num eeg priors: 15396
Num spectrogram priors: 10024
Num patient priors: 1910


In [14]:
np.set_printoptions(formatter={"all": lambda x: f"{x:0.3f}"})

import albumentations as A

spec_transforms = A.Compose([A.Resize(height=96, width=224)])


def proc_kspec(x):
    x = x.copy()

    x = x[:, 2:98]
    x[np.isnan(x) | np.isinf(x)] = 0
    x = x.clip(np.exp(-4), np.exp(7))
    x = np.log(x)

    x = x - x.mean(axis=(1, 2), keepdims=True)
    x = x / (x.std(axis=(1, 2), keepdims=True) + 1e-5)

    x = x.transpose(1, 2, 0)
    x = spec_transforms(image=x)["image"]
    x = x.transpose(2, 0, 1)

    x = x.reshape(4, 96, 224)
    return x


def _pad_or_crop_2d(x2d: np.ndarray, target_h: int, target_w: int) -> np.ndarray:
    h, w = x2d.shape
    if h > target_h:
        s = (h - target_h) // 2
        x2d = x2d[s : s + target_h, :]
    if w > target_w:
        s = (w - target_w) // 2
        x2d = x2d[:, s : s + target_w]
    h, w = x2d.shape
    if h < target_h or w < target_w:
        pad_h = target_h - h
        pad_w = target_w - w
        pad_top = pad_h // 2
        pad_bottom = pad_h - pad_top
        pad_left = pad_w // 2
        pad_right = pad_w - pad_left
        x2d = np.pad(x2d, ((pad_top, pad_bottom), (pad_left, pad_right)), mode="edge")
    return x2d


def proc_eeg_spec(x):
    x = x.copy()
    x[np.isnan(x) | np.isinf(x)] = 0
    x = x + 1.0

    out = np.empty((4, 96, 224), dtype=np.float32)
    for c in range(4):
        row = x[c].astype(np.float32)  # (T,)
        img = np.tile(row[None, :], (96, 1))  # (96, T)
        img = _pad_or_crop_2d(img, 96, 224)
        out[c] = img
    return out


def proc_eeg(eeg, mid, ekg):
    eeg = eeg.copy()
    mid = mid.copy()
    ekg = ekg.copy()

    eeg[np.isnan(eeg) | np.isinf(eeg)] = 0
    ekg[np.isnan(ekg) | np.isinf(ekg)] = 0
    mid[np.isnan(mid) | np.isinf(mid)] = 0

    eeg = eeg - eeg.mean(axis=-1, keepdims=True)
    mid = mid - mid.mean(axis=-1, keepdims=True)

    mad_std = MAD(eeg, axis=-1).reshape(-1)
    mad_std = np.median(mad_std) + 1e-5

    eeg = eeg / mad_std
    eeg = eeg.clip(-10, 10)

    mid = mid / mad_std
    mid = mid.clip(-10, 10)

    ekg = ekg / (MAD(ekg, axis=-1).reshape(-1) + 1e-5)

    eeg = eeg.reshape(16, -1)
    eeg = np.concatenate([eeg, mid, ekg], axis=0)

    eeg = eeg.reshape(19, 2_500)
    return eeg


# Keep caches/utilities in place (even though energy prior weight is 0.0 below).
_SPEC_ENERGY_CACHE: Dict[int, float] = {}


def _spec_energy_from_spectrogram_parquet(spec_parquet_path: str, spc_id: int) -> float:
    spc_id = int(spc_id)
    if spc_id in _SPEC_ENERGY_CACHE:
        return _SPEC_ENERGY_CACHE[spc_id]
    x = pl.read_parquet(spec_parquet_path).to_numpy().astype(np.float64)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    x = np.clip(x, np.exp(-4), np.exp(7))
    e = float(np.mean(np.log(x)))
    _SPEC_ENERGY_CACHE[spc_id] = e
    return e


def _build_energy_bin_priors_from_train(
    df_train: pl.DataFrame,
    spec_dir: str,
    global_prior: np.ndarray,
    n_bins: int = 16,
    max_unique_specs: int = 1800,
    alpha: float = 24.0,
) -> Tuple[np.ndarray, np.ndarray]:
    base = (
        df_train.group_by(["spectrogram_id", "label_id"])
        .agg([pl.col(c).sum().alias(c) for c in LABELS])
        .group_by("spectrogram_id")
        .agg([pl.col(c).sum().alias(c) for c in LABELS])
    )
    spec_ids = base["spectrogram_id"].to_list()
    if len(spec_ids) > max_unique_specs:
        spec_ids = sorted(spec_ids)[:max_unique_specs]
        base = base.filter(pl.col("spectrogram_id").is_in(spec_ids))

    vote_mat = base.select(LABELS).to_numpy().astype(np.float64)
    vote_mat = np.nan_to_num(vote_mat, nan=0.0, posinf=0.0, neginf=0.0)
    vote_mat = np.clip(vote_mat, 0.0, None)

    energies = []
    priors = []
    for sid, v in tqdm(
        zip(base["spectrogram_id"].to_list(), vote_mat),
        total=len(vote_mat),
        desc="Calibrating energy priors",
    ):
        sp = os.path.join(spec_dir, f"{int(sid)}.parquet")
        if not os.path.exists(sp):
            continue
        e = _spec_energy_from_spectrogram_parquet(sp, int(sid))
        p = v + alpha * global_prior
        p = _safe_row_normalize(p).reshape(-1)
        energies.append(e)
        priors.append(p)

    energies = np.asarray(energies, dtype=np.float64)
    priors = np.asarray(priors, dtype=np.float64)
    if len(energies) < 50:
        edges = np.linspace(-10.0, 10.0, n_bins + 1)
        table = np.tile(global_prior.reshape(1, -1), (n_bins, 1))
        return edges, table

    edges = np.quantile(energies, np.linspace(0, 1, n_bins + 1))
    edges = np.maximum.accumulate(edges)

    table = np.zeros((n_bins, 6), dtype=np.float64)

    bin_idx = np.clip(np.searchsorted(edges, energies, side="right") - 1, 0, n_bins - 1)
    for b in range(n_bins):
        m = bin_idx == b
        if not np.any(m):
            table[b] = global_prior
        else:
            table[b] = _safe_row_normalize(priors[m].mean(axis=0)).reshape(-1)

    lam = 0.25
    table = _safe_row_normalize((1.0 - lam) * table + lam * global_prior.reshape(1, -1))
    return edges, table


ENERGY_BIN_EDGES, ENERGY_BIN_PRIOR_TABLE = _build_energy_bin_priors_from_train(
    df_train=df_train,
    spec_dir=os.path.join(DATA_DIR, "train_spectrograms/"),
    global_prior=GLOBAL_PRIOR,
    n_bins=16,
    max_unique_specs=1800,
    alpha=24.0,
)

print("Energy prior table shape:", ENERGY_BIN_PRIOR_TABLE.shape)


def _energy_prior_for_spec_id(spc_id: int, spec_dir: str) -> np.ndarray:
    sp = os.path.join(spec_dir, f"{int(spc_id)}.parquet")
    if not os.path.exists(sp):
        return GLOBAL_PRIOR
    e = _spec_energy_from_spectrogram_parquet(sp, int(spc_id))
    b = int(
        np.clip(
            np.searchsorted(ENERGY_BIN_EDGES, e, side="right") - 1,
            0,
            ENERGY_BIN_PRIOR_TABLE.shape[0] - 1,
        )
    )
    return ENERGY_BIN_PRIOR_TABLE[b].reshape(-1)


def _mix_priors_for_row(eeg_id: int, spc_id: int, patient_id: int) -> np.ndarray:
    p0 = GLOBAL_PRIOR

    p_eeg = EEG_PRIORS.get(int(eeg_id), None)
    p_spc = SPEC_PRIORS.get(int(spc_id), None)
    p_pat = PATIENT_PRIORS.get(int(patient_id), None)

    c_eeg = EEG_COUNT.get(int(eeg_id), 0.0)
    c_spc = SPEC_COUNT.get(int(spc_id), 0.0)
    c_pat = PATIENT_COUNT.get(int(patient_id), 0.0)

    p_eng = _energy_prior_for_spec_id(int(spc_id), SPEC_DIR)

    # Change (score-relevant, fallback-only): make ID priors slightly more influential
    # to reduce underfitting (too-global / too-uniform), which tends to worsen KL.
    # We do this by modestly lowering shrinkage taus and allowing a bit more non-global mass.
    tau_eeg, tau_spc, tau_pat = 1600.0, 1200.0, 2200.0
    w_eeg = (c_eeg / (c_eeg + tau_eeg)) if p_eeg is not None else 0.0
    w_spc = (c_spc / (c_spc + tau_spc)) if p_spc is not None else 0.0
    w_pat = (c_pat / (c_pat + tau_pat)) if p_pat is not None else 0.0

    # Keep energy gating disabled (unchanged evaluation semantics from your current run).
    w_eng = 0.0

    w_sum = w_eeg + w_spc + w_pat + w_eng

    # Change (score-relevant, fallback-only): allow slightly more non-global weight
    # so the hierarchical priors can adapt more to IDs when evidence exists.
    max_non_global = 0.45
    if w_sum > max_non_global and w_sum > 0:
        scale = max_non_global / w_sum
        w_eeg *= scale
        w_spc *= scale
        w_pat *= scale
        w_eng *= scale

    w0 = 1.0 - (w_eeg + w_spc + w_pat + w_eng)

    p = w0 * p0
    if p_eeg is not None:
        p = p + w_eeg * p_eeg
    if p_spc is not None:
        p = p + w_spc * p_spc
    if p_pat is not None:
        p = p + w_pat * p_pat
    if w_eng > 0:
        p = p + w_eng * p_eng

    # Change (score-relevant, fallback-only): reduce uniform blending (was 8%),
    # which can make predictions too flat and increase KL; keep a small floor for safety.
    u = np.full(6, 1.0 / 6.0, dtype=np.float64)
    p = np.asarray(p, dtype=np.float64)
    p = 0.97 * p + 0.03 * u
    return _safe_row_normalize(p).reshape(-1)


@torch.no_grad()
def gen_ensemble_pred(df_row: pl.DataFrame) -> np.ndarray:
    eeg_id = df_row["eeg_id"].item()
    spc_id = df_row["spectrogram_id"].item()
    patient_id = int(df_row["patient_id"].item())

    eeg_filepath = os.path.join(EEG_DIR, f"{eeg_id}.parquet")
    spc_filepath = os.path.join(SPEC_DIR, f"{spc_id}.parquet")

    if len(models_multimodal) == 0:
        pred = _mix_priors_for_row(int(eeg_id), int(spc_id), int(patient_id))
        pred = (
            _safe_row_normalize(np.asarray(pred, dtype=np.float64)).reshape(-1).copy()
        )
        return pred

    preds = []

    kspec = compute_kaggle_spec_from_file(spc_filepath)
    kspec = proc_kspec(kspec)
    kspec = torch.tensor(kspec, dtype=torch.float32, device=device).unsqueeze(0)

    eeg_spec = compute_spec_from_file(eeg_filepath)
    eeg_spec = proc_eeg_spec(eeg_spec)
    eeg_spec = torch.tensor(eeg_spec, dtype=torch.float32, device=device).unsqueeze(0)

    eeg, mid, ekg = compute_eeg_from_file(eeg_filepath)
    eeg[np.isnan(eeg) | np.isinf(eeg)] = 0
    eeg = proc_eeg(eeg, mid, ekg)
    eeg = torch.tensor(eeg, dtype=torch.float32, device=device).unsqueeze(0)

    for model in models_multimodal:
        model.eval()
        out = model(eeg, eeg_spec, kspec)
        pred = out.exp().detach().float().cpu().numpy().reshape(-1)
        preds.append(pred)

    preds = np.mean(np.stack(preds, axis=0), axis=0)
    preds = _safe_row_normalize(preds).reshape(-1)
    return preds.astype(np.float64)




Calibrating energy priors:   0%|          | 0/1800 [00:00<?, ?it/s]

Energy prior table shape: (16, 6)


In [15]:
from tqdm.auto import tqdm

preds_final = []
for i in tqdm(range(len(df_test))):
    pred = gen_ensemble_pred(df_test[i])
    preds_final.append(pred)

preds_final = np.asarray(preds_final, dtype=np.float64)
print("preds_final shape:", preds_final.shape)
print(
    "row sums (min/max):", preds_final.sum(axis=1).min(), preds_final.sum(axis=1).max()
)



  0%|          | 0/9850 [00:00<?, ?it/s]

preds_final shape: (9850, 6)
row sums (min/max): 0.9999999999999998 1.0000000000000002


In [16]:
eeg_ids = df_test["eeg_id"].to_list()
df_sub = pd.DataFrame({"eeg_id": eeg_ids})

preds_final = np.nan_to_num(preds_final, nan=0.0, posinf=0.0, neginf=0.0)
preds_final = np.clip(preds_final, 1e-12, None)
preds_final = preds_final / preds_final.sum(axis=1, keepdims=True)

df_sub[LABELS] = preds_final
df_sub.to_csv("submission.csv", index=False)

print(df_sub.head())
print("Wrote submission.csv with shape:", df_sub.shape)
print(
    "Check row sums:",
    float(df_sub[LABELS].sum(axis=1).min()),
    float(df_sub[LABELS].sum(axis=1).max()),
)

       eeg_id  seizure_vote  lpd_vote  gpd_vote  lrda_vote  grda_vote  \
0  2578018731       0.09837  0.098748  0.335969   0.095202   0.175697   
1  2578018731       0.09837  0.098748  0.335969   0.095202   0.175697   
2  2578018731       0.09837  0.098748  0.335969   0.095202   0.175697   
3  2578018731       0.09837  0.098748  0.335969   0.095202   0.175697   
4  1706196508       0.15281  0.208879  0.108224   0.154949   0.118053   

   other_vote  
0    0.196015  
1    0.196015  
2    0.196015  
3    0.196015  
4    0.257085  
Wrote submission.csv with shape: (9850, 7)
Check row sums: 0.9999999999999998 1.0000000000000002
